# 02 — Universal Preprocessing\n\n
Standardize raw audio to a common format for downstream classical and neural pipelines: resample to 16 kHz, high-pass at 80 Hz, peak normalize, segment into 1 s windows with 50% overlap, and optionally create subtle augmentations (noise, time stretch, pitch shift, gain).

In [27]:
from pathlib import Path
import yaml
import random
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.auto import tqdm
from scipy.signal import butter, filtfilt  # For HPF; pip install scipy if missing
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
CFG_PATH = PROJECT_ROOT / 'config.yaml'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
OUT_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'universal'
METRICS_DIR = PROJECT_ROOT / 'results' / 'metrics'
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# Seed for reproducibility
np.random.seed(42)
random.seed(42)

# Load config with fallbacks
with open(CFG_PATH, 'r') as f:
    yaml_cfg = yaml.safe_load(f)

audio_cfg = yaml_cfg.get('audio', {})
aug_cfg = audio_cfg.get('augmentation', {})  
target_sr = int(audio_cfg.get('sample_rate', 16000))
win_dur = float(audio_cfg.get('window_duration', 1.0))
overlap = float(audio_cfg.get('overlap_fraction', 0.5))  # Ensure float
hp_cut = float(audio_cfg.get('highpass_cutoff', 80))
norm_mode = audio_cfg.get('normalization_mode', 'peak')
target_peak = float(audio_cfg.get('target_peak_level', 0.95))

# Aug defaults (ensure all keys)
aug_defaults = {
    'prob_background': 0.0,
    'prob_minority': 0.8,
    'n_variants_per_file': 2,
    'noise_snr_db_range': (15, 30),
    'time_stretch_range': (0.95, 1.05),
    'pitch_shift_semitones': (-1, 1),
    'gain_range': (0.8, 1.2),
}
aug_cfg.update({k: v for k, v in aug_defaults.items() if k not in aug_cfg})

print('Config loaded:', {'sr': target_sr, 'win_dur': win_dur, 'overlap': overlap, 'hp_cut': hp_cut, 'aug_prob_bg': aug_cfg['prob_background'], 'aug_prob_min': aug_cfg['prob_minority']})


Config loaded: {'sr': 16000, 'win_dur': 1.0, 'overlap': 0.5, 'hp_cut': 80.0, 'aug_prob_bg': 0.0, 'aug_prob_min': 0.8}


## Utilities

In [28]:
def load_audio(path, target_sr):
    """Load and resample audio with validation."""
    try:
        y, sr = librosa.load(path, sr=target_sr, mono=True)
        if y is None or len(y) == 0 or not np.issubdtype(y.dtype, np.number):
            raise ValueError("Empty or invalid audio array")
        return y, sr
    except Exception as e:
        print(f'  Load failed for {path.name}: {e} (skipping)')
        return None, None

def apply_highpass(y, sr, cutoff):
    """High-pass filter with validation."""
    if y is None or len(y) == 0:
        return None
    try:
        nyq = 0.5 * sr
        normal_cutoff = cutoff / nyq
        b, a = butter(4, normal_cutoff, btype='high', analog=False)
        return filtfilt(b, a, y)
    except:
        return y  # Fallback to original if filter fails

def normalize_audio(y, mode='peak', target_peak=0.95):
    """Peak normalization with validation."""
    if y is None or len(y) == 0:
        return None
    try:
        if mode == 'peak':
            peak = np.max(np.abs(y))
            if peak > 0:
                y = y * (target_peak / peak)
        return y
    except:
        return y

def segment_audio(y, sr, win_dur=1.0, overlap=0.5):
    """Segment into overlapping windows with validation."""
    if y is None or len(y) == 0:
        return []
    try:
        frame_length = int(win_dur * sr)
        hop_length = int(frame_length * (1 - overlap))
        if hop_length <= 0:
            raise ValueError("Invalid hop_length")
        frames = librosa.util.frame(y, frame_length=frame_length, hop_length=hop_length)
        return [frame.flatten() for frame in frames.T if len(frame.flatten()) == frame_length]  # Valid 1s only
    except Exception as e:
        print(f'  Segmentation failed: {e}')
        return []


## Augmentations (subtle)

In [29]:
import random
import numpy as np
import librosa

def add_white_noise(y, snr_db_range=(15, 30)):
    """Add white noise at target SNR (dB)."""
    snr_db = random.uniform(*snr_db_range)
    sig_power = np.mean(y**2) + 1e-9
    noise_power = sig_power / (10 ** (snr_db / 10.0))
    noise = np.random.normal(0, np.sqrt(noise_power), size=y.shape)
    return y + noise, {'noise_snr_db': snr_db}

def time_stretch(y, rate_range=(0.95, 1.05)):
    """Subtle time stretch (preserves pitch)."""
    rate = random.uniform(*rate_range)
    y_st = librosa.effects.time_stretch(y, rate=rate)
    return y_st, {'time_stretch_rate': rate}

def pitch_shift(y, sr, semitone_range=(-1, 1)):
    """Subtle pitch shift (±1 semitone)."""
    steps = random.uniform(*semitone_range)
    y_ps = librosa.effects.pitch_shift(y, sr=sr, n_steps=steps)
    return y_ps, {'pitch_semitones': steps}

def apply_gain(y, gain_range=(0.8, 1.2)):
    """Linear gain adjustment."""
    g = random.uniform(*gain_range)
    return y * g, {'gain_factor': g}

def maybe_augment_single(y, sr, aug_cfg):
    """Apply 1-2 random augmentations to a single variant (core logic)."""
    ops = [
        lambda z: add_white_noise(z, aug_cfg.get('noise_snr_db_range', (15, 30))),
        lambda z: time_stretch(z, aug_cfg.get('time_stretch_range', (0.95, 1.05))),
        lambda z: pitch_shift(z, sr, aug_cfg.get('pitch_shift_semitones', (-1, 1))),
        lambda z: apply_gain(z, aug_cfg.get('gain_range', (0.8, 1.2))),
    ]
    k = random.choice([1, 2])  # 1-2 ops per variant
    params = {}
    z = y.copy()
    for fn in random.sample(ops, k=k):
        z, p = fn(z)
        params.update(p)
    return z, params

def generate_augmented_variants(y, sr, label, aug_cfg):
    """
    Generate augmented variants for a full clip, class-aware.
    Returns: list of signals, list of param dicts (for manifest).
    """
    random.seed(hash(f"{label}_{hash(str(y[:10]))}") % (2**32))  # Reproducible per clip
    variants = [y.copy()]  # Original always
    variant_params = [{}]   # Empty for original

    # Class-specific prob
    p_apply = aug_cfg.get('prob_background', 0.0) if label == 'background' else aug_cfg.get('prob_minority', 0.8)

    if random.random() < p_apply and p_apply > 0:
        n_variants = random.randint(1, aug_cfg.get('n_variants_per_file', 2))
        for _ in range(n_variants):
            aug_y, params = maybe_augment_single(y, sr, aug_cfg)
            variants.append(aug_y)
            variant_params.append(params)

    print(f'  {label}: {len(variants)-1} aug variants (prob={p_apply})')
    return variants, variant_params


## Processing pipeline

In [30]:
def process_file(path, label, target_sr=16000, win_dur=1.0, overlap=0.5, hp_cut=80, 
                 norm_mode='peak', target_peak=0.95, out_root=None, aug_cfg=None):
    """Process with robust error handling."""
    if aug_cfg is None:
        aug_cfg = {}
    
    records = []
    try:
        # Load
        y, sr = load_audio(path, target_sr)
        if y is None or len(y) == 0:
            return records  # Skip empty
        
        orig_dur = len(y) / sr  # Safe now
        
        # Preprocess
        y = apply_highpass(y, sr, hp_cut)
        if y is None:
            return records
        y = normalize_audio(y, norm_mode, target_peak)
        if y is None:
            return records
        
        # Generate variants
        variants, variant_params = generate_augmented_variants(y, sr, label, aug_cfg)
        if not variants:
            return records  # No variants (rare)
        
        out_dir = out_root / label
        out_dir.mkdir(parents=True, exist_ok=True)
        
        for v_idx, (var_y, v_params) in enumerate(zip(variants, variant_params)):
            # Normalize post-aug
            var_y = normalize_audio(var_y, norm_mode, target_peak)
            if var_y is None:
                continue
                
            # Segment
            segments = segment_audio(var_y, sr, win_dur, overlap)
            if not segments:
                continue
            
            v_dur = len(var_y) / sr
            
            for s_idx, seg in enumerate(segments):
                # Safe naming
                stem = Path(path).stem
                if v_idx == 0:
                    suffix = f'original_{s_idx}_{win_dur:.1f}s.wav'
                else:
                    aug_keys = [k for k in v_params if v_params.get(k)]
                    aug_desc = '_'.join(aug_keys[:2]) if aug_keys else 'aug'
                    suffix = f'{aug_desc}_{v_idx}_{s_idx}_{win_dur:.1f}s.wav'
                
                seg_path = out_dir / f'{stem}_{suffix}'
                sf.write(seg_path, seg, target_sr)
                
                # Record
                records.append({
                    'original_path': str(path),
                    'segment_path': str(seg_path),
                    'label': label,
                    'original_duration_s': float(orig_dur),
                    'variant_idx': int(v_idx),
                    'segment_idx': int(s_idx),
                    'segment_duration_s': float(win_dur),
                    'sample_rate': int(target_sr),
                    'aug_params': v_params,  # Dict
                    'peak_level': float(np.max(np.abs(seg))),
                    'rms_level': float(np.sqrt(np.mean(seg**2))),
                })
        
        return records
    except Exception as e:
        print(f'Error processing {path.name}: {e}')
        return records  # Empty list

In [31]:
CLASSES = ['background', 'angle_grinder', 'tools']
records = []
valid_files = 0

for label in CLASSES:
    label_dir = RAW_DIR / label
    if not label_dir.exists():
        print(f'Skipping {label}: dir not found')
        continue
    
    # Filter valid audio files (skip non-audio if needed)
    files = [p for p in label_dir.rglob('*') if p.suffix.lower() in ['.wav', '.mp3', '.flac']]
    print(f'Processing {label}: {len(files)} files')
    
    for p in tqdm(files, desc=label):
        entries = process_file(
            p, label, target_sr, win_dur, overlap, hp_cut, 
            norm_mode, target_peak, OUT_ROOT, aug_cfg
        )
        records.extend(entries)
        if entries:  # Count valid
            valid_files += 1

print(f'Processed {valid_files} valid files out of total attempted')

# Safe manifest creation
if records:
    manifest = pd.DataFrame.from_records(records)
    if 'aug_params' in manifest.columns:
        manifest['aug_params'] = manifest['aug_params'].apply(lambda d: str(d) if isinstance(d, dict) else str(d))
    else:
        manifest['aug_params'] = '{}'  # Default empty
    
    # Ensure numeric types
    numeric_cols = ['original_duration_s', 'segment_duration_s', 'peak_level', 'rms_level', 'variant_idx', 'segment_idx', 'sample_rate']
    for col in numeric_cols:
        if col in manifest.columns:
            manifest[col] = pd.to_numeric(manifest[col], errors='coerce')
    
    manifest.to_csv(METRICS_DIR / 'preprocessing_manifest.csv', index=False)
    print(f'Saved manifest with {len(manifest)} segments')
    
    # Stats
    print(manifest.groupby('label').agg({
        'segment_path': 'count',
        'original_path': 'nunique',  # Unique files per class
        'variant_idx': 'max'  # Max variants used
    }).round(0))
else:
    print('No valid records generated—check raw data/loading issues')


Processing background: 1348 files


background:   0%|          | 0/1348 [00:00<?, ?it/s]

  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)
  background: 0 aug variants (prob=0.0)


angle_grinder:   0%|          | 0/36 [00:00<?, ?it/s]

  angle_grinder: 2 aug variants (prob=0.8)
  angle_grinder: 1 aug variants (prob=0.8)
  angle_grinder: 0 aug variants (prob=0.8)
  angle_grinder: 0 aug variants (prob=0.8)
  angle_grinder: 1 aug variants (prob=0.8)
  angle_grinder: 2 aug variants (prob=0.8)
  angle_grinder: 1 aug variants (prob=0.8)
  angle_grinder: 0 aug variants (prob=0.8)
  angle_grinder: 2 aug variants (prob=0.8)
  angle_grinder: 2 aug variants (prob=0.8)
  angle_grinder: 1 aug variants (prob=0.8)
  angle_grinder: 0 aug variants (prob=0.8)
  angle_grinder: 1 aug variants (prob=0.8)
  angle_grinder: 2 aug variants (prob=0.8)
  angle_grinder: 0 aug variants (prob=0.8)
  angle_grinder: 1 aug variants (prob=0.8)
  angle_grinder: 2 aug variants (prob=0.8)
  angle_grinder: 2 aug variants (prob=0.8)
  angle_grinder: 0 aug variants (prob=0.8)
  angle_grinder: 2 aug variants (prob=0.8)
  angle_grinder: 1 aug variants (prob=0.8)
  angle_grinder: 2 aug variants (prob=0.8)
  angle_grinder: 1 aug variants (prob=0.8)
  angle_gri

tools:   0%|          | 0/21 [00:00<?, ?it/s]

  tools: 1 aug variants (prob=0.8)
  tools: 1 aug variants (prob=0.8)
  tools: 1 aug variants (prob=0.8)
  tools: 1 aug variants (prob=0.8)
  tools: 0 aug variants (prob=0.8)
  tools: 2 aug variants (prob=0.8)
  tools: 1 aug variants (prob=0.8)
  tools: 2 aug variants (prob=0.8)
  tools: 1 aug variants (prob=0.8)
  tools: 1 aug variants (prob=0.8)
  tools: 2 aug variants (prob=0.8)
  tools: 2 aug variants (prob=0.8)
  tools: 2 aug variants (prob=0.8)
  tools: 1 aug variants (prob=0.8)
  tools: 0 aug variants (prob=0.8)
  tools: 2 aug variants (prob=0.8)
  tools: 2 aug variants (prob=0.8)
  tools: 1 aug variants (prob=0.8)
  tools: 1 aug variants (prob=0.8)
  tools: 1 aug variants (prob=0.8)
  tools: 0 aug variants (prob=0.8)
Processed 1405 valid files out of total attempted
Saved manifest with 19124 segments
               segment_path  original_path  variant_idx
label                                                  
angle_grinder          6458             36            2
background  

## Spot-check examples

In [33]:
import matplotlib.pyplot as plt
import librosa.display as ld  # Add to imports if missing

CLASSES = ['background', 'angle_grinder', 'tools']  # Your classes
FIG_DIR = PROJECT_ROOT / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

if manifest.empty:
    print('Manifest empty—no segments to plot. Check processing logs.')
else:
    examples = []
    for cls in CLASSES:
        cls_mask = manifest['label'] == cls
        if cls_mask.any():
            # Get first valid segment path
            ex_path = manifest[cls_mask]['segment_path'].head(1).iloc[0]
            if pd.notna(ex_path) and Path(ex_path).exists():
                examples.append(ex_path)
            else:
                print(f'No valid path for {cls}')
        else:
            print(f'No segments for class {cls}')
    
    if examples:
        print(f'Plotting examples from {len(examples)} classes')
        for p in examples:
            try:
                y, sr = sf.read(p)
                # Trim if too long for plot (e.g., first 5s)
                if len(y) > sr * 5:
                    y = y[:sr * 5]
                
                fig, ax = plt.subplots(2, 1, figsize=(10, 6))
                
                # Waveform
                ld.waveshow(y, sr=sr, ax=ax[0], color='steelblue')
                ax[0].set_title(f'Waveform: {Path(p).name}', fontsize=12)
                ax[0].set_ylabel('Amplitude')
                
                # Spectrogram (log-freq, STFT)
                S = librosa.stft(y, n_fft=1024, hop_length=256)
                S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)
                img = ld.specshow(S_db, sr=sr, hop_length=256, x_axis='time', y_axis='log', 
                                  ax=ax[1], cmap='magma')
                ax[1].set_title('Mel Spectrogram (log-freq, dB)', fontsize=12)
                ax[1].set_ylabel('Frequency (Hz)')
                fig.colorbar(img, ax=ax[1], format='%+2.0f dB')
                
                plt.tight_layout()
                
                # Save
                stem = Path(p).stem
                out_fig = FIG_DIR / f'spotcheck_{stem}.png'
                plt.savefig(out_fig, dpi=150, bbox_inches='tight')
                plt.close(fig)
                
                print(f'Saved: {out_fig}')
                
            except Exception as e:
                print(f'Plot failed for {p}: {e}')
    else:
        print('No valid examples found—verify segment paths exist')


Plotting examples from 3 classes
Saved: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/results/figures/spotcheck_chunk_1304_original_0_1.0s.png
Saved: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/results/figures/spotcheck_412255__mehdiikazemi__angle-grinder_original_0_1.0s.png
Saved: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/results/figures/spotcheck_drill-200178_original_0_1.0s.png
